In [7]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2"

In [8]:
import torch
from custom_vision_transformer import vit_custom
from training_utils import *
from custom_vision_transformer import *
import matplotlib.pyplot as plt
from types import SimpleNamespace

In [9]:
params = SimpleNamespace(
    gpu_idx=0,
    num_classes=100,
    img_size=224,
    batch_size=16,
)

In [10]:
print("Number of visible GPUs :", torch.cuda.device_count())

torch.cuda.set_device(params.gpu_idx)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}:{torch.cuda.current_device()} {torch.cuda.get_device_name(torch.cuda.current_device())}" if torch.cuda.is_available() else "CPU")

Number of visible GPUs : 3
Device: cuda:0 NVIDIA GeForce RTX 2080


In [11]:
# --- load model ---
model = vit_custom(num_classes=params.num_classes, image_size=params.img_size, attention_bias=True).to(device)
# model.load_state_dict(torch.load("./training_checkpoints/vit_custom_16_epoch_60.pth"))
model.eval();

In [ ]:
train_loader, val_loader = get_data_loaders(batch_size=params.batch_size, persistent_workers=False, shuffle_val=True)

In [15]:
imgs, labels = next(iter(val_loader))

In [16]:
# First, make sure all hooks are removed
for module in model.modules():
    module._forward_hooks.clear()

In [17]:
preds1 = model(imgs.to(device))
print(preds1.shape)

torch.Size([16, 100])


In [18]:
top_k = 150 # total is 197

for module in model.modules(): # Nettoyer tous les hooks existants sur tous les modules
    module._forward_hooks.clear()

def hook(module, input, output):
    out, attn = output  # (B, H, T, D), (B, H, T, T)
    x = input[0]  # (B, T, D)
    B, T, C = x.shape

    # --- recompute V ---
    qkv = module.qkv(x).reshape(B, T, 3, module.num_heads, module.head_dim)
    _, _, v = qkv.unbind(2)   # (B, T, H, D)
    v = v.transpose(1, 2)     # (B, H, T, D)

    # --- mask tokens (example: keep top_k tokens per head) ---
    token_importance = attn.mean(dim=-2)  # (B, H, T) Average over query tokens
    mask = torch.zeros_like(token_importance, dtype=torch.bool)
    for b in range(B):
        for h in range(module.num_heads):
            topk_idx = torch.topk(token_importance[b,h], k=top_k).indices
            mask[b,h,topk_idx] = True

    attn = attn.clone()
    attn[~mask.unsqueeze(-2).expand(-1, -1, T, -1)] = 0.0
    attn = attn / (attn.sum(dim=-1, keepdim=True) + 1e-8)

    # --- recompute output ---
    out = attn @ v
    out = out.transpose(1,2).reshape(B, T, C)
    out = module.out_proj(out)

    return out, attn

In [19]:
for name, module in model.named_modules():
    if name.endswith(".attention"):
        module.register_forward_hook(hook)

In [20]:
preds2 = model(imgs.to(device))
print(preds2.shape)

torch.Size([16, 100])


In [21]:
print(preds1[0])
print(preds2[0])
print("Difference in predictions:", (preds1 - preds2).abs().mean().item())
print(preds1.shape)

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.], device='cuda:0', grad_fn=<SelectBackward0>)
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.], device='cuda:0', grad_fn=<SelectBackward0>)
Difference in predictions: 0.0